<a href="https://colab.research.google.com/github/agbizbuz/learning-ai-ds-ml/blob/main/Course_Work/continued_pretraining_engineering_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Continued Pretraining Demo for Engineering Text

This notebook demonstrates a **tiny continued pretraining workflow** for an engineering use case.

## Learning flow
1. Load a **small pretrained/base model**
2. Test it on a generic engineering prompt
3. Create a tiny engineering corpus
4. Perform **continued pretraining**
5. Compare output **before vs after**

## Important note
This demo is not a production training pipeline.

## 1. Install dependencies
Run this cell first in Google Colab.

In [ ]:
!pip -q install transformers datasets accelerate sentencepiece

## 2. Imports

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

## 3. Load a small pretrained model

We use a small model so the demo can run quickly in Colab.

You can try:
- `distilgpt2` for a small practical demo
- `sshleifer/tiny-gpt2` for an even faster but weaker demo

In [ ]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded model:", model_name)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded model: distilgpt2


## 4. Define a helper for text generation

In [ ]:
def generate_text(prompt, max_new_tokens=40, temperature=0.8):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## 5. Baseline generation before continued pretraining

This shows what the pretrained/base model knows **before** seeing engineering-specific training text.

In [ ]:
prompt = "A centrifugal pump in the cooling system showed abnormal vibration because"
print("=== BEFORE CONTINUED PRETRAINING ===")
print(generate_text(prompt))

=== BEFORE CONTINUED PRETRAINING ===
A centrifugal pump in the cooling system showed abnormal vibration because the centrifugal pump was not engaged in a proper mechanism during the cooling process.





The centrifugal pump was not properly engaged in a proper mode during the cooling process. However


In [ ]:
def generate_text(prompt, max_new_tokens=40):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "def find_max(numbers):"
print("=== BEFORE CONTINUED PRETRAINING ===")
print(generate_text(prompt))

=== BEFORE CONTINUED PRETRAINING ===
def find_max(numbers): [ 'numbers', 'numbers') if (numbers)) return numbers+1























## 6. Create a tiny engineering corpus

This is a **small domain corpus** related to industrial equipment, pumps, vibration, and maintenance.
In real pretraining, the dataset would be much larger.

In [ ]:
engineering_texts = [
    "A centrifugal pump in the cooling system started showing abnormal vibration levels during operation.",
    "The vibration amplitude increased whenever the pump operated above 2800 RPM.",
    "This pattern usually indicates bearing wear or shaft misalignment.",
    "Early detection of anomalies helps prevent equipment failure and unplanned downtime.",
    "Misalignment in rotating equipment can increase heat, vibration, and energy loss.",
    "Bearing wear is a common cause of abnormal vibration in pumps and motors.",
    "Condition monitoring helps maintenance engineers detect faults before breakdown.",
    "Industrial equipment diagnostics often rely on vibration, temperature, and pressure trends.",
    "A maintenance engineer checks shaft alignment when vibration exceeds threshold values.",
    "Predictive maintenance reduces downtime and improves asset reliability in plants."
]

dataset = Dataset.from_dict({"text": engineering_texts})
dataset

Dataset({
    features: ['text'],
    num_rows: 10
})

## 7. Tokenize the dataset for causal language modeling

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

tokenized_dataset

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 10
})

## 8. Training configuration

This is a **tiny continued pretraining setup** for demonstration only.
You can reduce `num_train_epochs` if Colab runtime is slow.

In [ ]:
training_args = TrainingArguments(
    output_dir="./engineering_demo_model",
    num_train_epochs=20,
    per_device_train_batch_size=2,
    save_steps=50,
    save_total_limit=1,
    logging_steps=10,
    prediction_loss_only=True,
    report_to="none"
)

## 9. Continued pretraining

This is **not instruction fine-tuning**.  
This is a small example of **continued pretraining on engineering text**.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.988925
20,3.341343
30,2.297477
40,1.574505
50,1.156577
60,0.881307
70,0.787854
80,0.738469
90,0.713646
100,0.607979


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=1.7088082265853881, metrics={'train_runtime': 173.1006, 'train_samples_per_second': 1.155, 'train_steps_per_second': 0.578, 'total_flos': 3266209382400.0, 'train_loss': 1.7088082265853881, 'epoch': 20.0})

## 10. Generate again after continued pretraining

Now compare the output with the earlier result.
Look for more engineering-specific words such as:
- bearing wear
- shaft misalignment
- vibration
- predictive maintenance

In [ ]:
print("=== AFTER CONTINUED PRETRAINING ===")
prompt = "A centrifugal pump in the cooling system showed abnormal vibration because"
print(generate_text(prompt))

=== AFTER CONTINUED PRETRAINING ===
A centrifugal pump in the cooling system showed abnormal vibration because of vibration during operation. Electrical faults in the cooling system often exceed normal vibration levels. In addition, abnormal vibration vibration can increase heat, vibration, temperature, and pressure.








## 11. Try a second engineering prompt

In [ ]:
prompt2 = "Abnormal vibration in a pump may indicate"
print("=== SECOND PROMPT AFTER CONTINUED PRETRAINING ===")
print(generate_text(prompt2))

=== SECOND PROMPT AFTER CONTINUED PRETRAINING ===
Abnormal vibration in a pump may indicate abnormal vibration levels during pumping.




































## 12. Reflection

### What this notebook demonstrates
- A **pretrained/base model** can generate general text
- If domain knowledge is limited, the output may be generic
- **Continued pretraining** helps the model become more domain-aware

### What this notebook does not demonstrate
- Large-scale industrial pretraining
- Production-grade evaluation
- Instruction tuning or alignment

## 13. Key Points

> Add blockquote



- **"This is a base model."**
- **"The model is fluent, but may not be engineering-aware."**
- **"This is the miniature version of continued pretraining for domain adaptation."**